# 🎓 Educational Tutorial: AlexNet Architecture from Scratch

This notebook serves as a self-contained learning artifact detailing the history, design choices, mathematical intuition, and implementation of **AlexNet** (2012) from first principles.

---

## 1. Original Paper & History

* **Title**: *ImageNet Classification with Deep Convolutional Neural Networks*
* **Authors**: Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton
* **Year**: 2012 (Presented at NIPS/NeurIPS)

### Historical Context
Prior to 2012, the field of computer vision was dominated by classical hand-crafted features (such as SIFT, HOG, and LBP) combined with support vector machines (SVMs) or random forests. The ImageNet Large Scale Visual Recognition Challenge (ILSVRC), containing over 1.2 million high-resolution images across 1,000 categories, was the ultimate test. In 2012, AlexNet won the competition by achieving a top-5 error rate of **15.3%**, whereas the runner-up (using classical features) achieved **26.2%**. This watershed event marked the beginning of the modern Deep Learning boom.

## 2. Problem Being Solved

Prior networks (like LeNet-5) proved effective on small, controlled classification tasks like MNIST digit recognition (28x28 grayscale digits). However, scaling up to complex real-world image datasets presented several major bottlenecks:

1. **Computational Limitations**: Training deep neural networks on millions of large high-resolution images required trillions of floating-point operations. AlexNet solved this by implementing parallelized convolutions across two NVIDIA GTX 580 GPUs.
2. **Overfitting on High-Capacity Models**: Deeper networks with millions of parameters quickly memorized the training dataset. AlexNet introduced two major regularizers: **Dropout** and heavy **Data Augmentation**.
3. **Vanishing Gradients**: Deep networks using sigmoid or tanh activations suffered from saturated gradients, stalling training. AlexNet introduced the **Rectified Linear Unit (ReLU)** activation function, accelerating convergence.

## 3. How AlexNet Differs from LeNet-5

AlexNet is structurally a scaled-up successor to LeNet-5, but it differs in several fundamental ways:

| Feature | LeNet-5 (1998) | AlexNet (2012) |
|---|---|---|
| **Input Resolution** | 32 × 32 (Grayscale) | 224 × 224 × 3 (RGB) |
| **Depth** | 2 Conv Layers, 3 FC Layers | 5 Conv Layers, 3 FC Layers |
| **Parameters** | ~60,000 | ~60 Million |
| **Non-Linearity** | Tanh / Sigmoid | Rectified Linear Unit (ReLU) |
| **Pooling** | Average Pooling (2x2) | Max Pooling (3x3 with stride 2 - overlapping) |
| **Regularization** | None | Dropout (0.5), Weight Decay, Augmentation |
| **Hardware** | Intel CPU | Dual NVIDIA GTX 580 GPUs |

## 4. Mathematical Intuition

### 4.1. Spatial Output Dimension Formula
For a convolutional layer with input size $W$, kernel size $K$, padding $P$, and stride $S$, the output size $O$ is given by:

$$O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

For example, in AlexNet's Layer 1:
* Input $W = 224$
* Kernel $K = 11$
* Padding $P = 2$
* Stride $S = 4$

$$O = \left\lfloor \frac{224 - 11 + 2(2)}{4} \right\rfloor + 1 = \left\lfloor \frac{217}{4} \right\rfloor + 1 = 54 + 1 = 55$$
Thus, the output feature map is $55 \times 55$ per channel.

### 4.2. ReLU Activation Function
ReLU thresholds all negative values to zero:

$$f(x) = \max(0, x)$$

Its derivative is:

$$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{if } x < 0 \end{cases}$$

Because the derivative is $1.0$ for positive inputs, the gradients flow freely backwards without being squashed or saturated (which happens with Sigmoid/Tanh where derivatives approach 0).

## 5. Layer-by-Layer Architectural Discussion

1. **Input**: $224 \times 224 \times 3$ image.
2. **Conv1**: $11 \times 11$ filters, stride 4, padding 2. Output: $55 \times 55 \times 64$.
3. **MaxPool1**: $3 \times 3$ kernel, stride 2 (overlapping pooling). Output: $27 \times 27 \times 64$.
4. **Conv2**: $5 \times 5$ filters, padding 2, stride 1. Output: $27 \times 27 \times 192$.
5. **MaxPool2**: $3 \times 3$ kernel, stride 2. Output: $13 \times 13 \times 192$.
6. **Conv3**: $3 \times 3$ filters, padding 1, stride 1. Output: $13 \times 13 \times 384$.
7. **Conv4**: $3 \times 3$ filters, padding 1, stride 1. Output: $13 \times 13 \times 256$.
8. **Conv5**: $3 \times 3$ filters, padding 1, stride 1. Output: $13 \times 13 \times 256$.
9. **MaxPool3**: $3 \times 3$ kernel, stride 2. Output: $6 \times 6 \times 256$.
10. **Fully Connected 1**: Flatten to $9216$ nodes -> Linear to $4096$ nodes. Dropout applied.
11. **Fully Connected 2**: Linear to $4096$ nodes. Dropout applied.
12. **Output**: Linear to $10$ nodes (representing the EuroSAT classes).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import time

print("PyTorch version:", torch.__version__)

In [ ]:
class AlexNetFromScratch(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        
        # 1. Feature Extractor
        self.features = nn.Sequential(
            # Conv 1
            nn.Conv2d(in_channels, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv 2
            nn.Conv2d(64, 192, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv 3
            nn.Conv2d(192, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv 4
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv 5
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Adaptive Average Pooling
            nn.AdaptiveAvgPool2d((6, 6))
        )
        
        # 2. Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            
            nn.Linear(4096, num_classes)
        )
        
        self._initialize_weights()
        
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

model = AlexNetFromScratch()
print("Parameter Count:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
class NotebookEuroSATDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = self.root_dir / row["image_path"]
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        
        if self.transform:
            image = self.transform(image)
            
        return {
            "image": image,
            "label": label
        }

In [ ]:
PROJECT_ROOT = Path("../../")  # Assumed relative path from notebooks/CNN_Evolution/
csv_path = PROJECT_ROOT / "data/processed/train.csv"
val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

if not csv_path.exists():
    # Check local folder to see if we can find it
    PROJECT_ROOT = Path(".").resolve().parents[1]
    csv_path = PROJECT_ROOT / "data/processed/train.csv"
    val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
    img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

print("CSV path:", csv_path.resolve())
print("Images directory:", img_dir.resolve())

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if csv_path.exists() and img_dir.exists():
    train_dataset = NotebookEuroSATDataset(csv_path, img_dir, train_transform)
    val_dataset = NotebookEuroSATDataset(val_csv_path, img_dir, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    print(f"Dataset loaded successfully. {len(train_dataset)} train samples, {len(val_dataset)} val samples.")
else:
    print("Warning: Dataset split CSV files or raw EuroSAT directories not found! Mocking data loading.")
    class DummyDataset(Dataset):
        def __len__(self): return 64
        def __getitem__(self, idx): return {"image": torch.randn(3, 224, 224), "label": torch.randint(0, 10, (1,)).item()}
    train_loader = DataLoader(DummyDataset(), batch_size=8, shuffle=True)
    val_loader = DataLoader(DummyDataset(), batch_size=8, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

model = AlexNetFromScratch().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Train for 1 quick epoch to show verification
model.train()
start_time = time.time()
epoch_loss = 0.0
correct = 0
total = 0

print("Starting Epoch 1...")
# To keep it fast in notebook validation, limit to first 5 batches if verifying
max_batches = 5
for idx, batch in enumerate(train_loader):
    if idx >= max_batches: break
    images = batch["image"].to(device)
    labels = batch["label"].to(device)
    
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    epoch_loss += loss.item() * images.size(0)
    correct += (outputs.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)
    
print(f"Epoch complete in {time.time() - start_time:.2f}s | Loss: {epoch_loss/total:.4f} | Acc: {correct/total*100:.2f}%")

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for idx, batch in enumerate(val_loader):
        if idx >= 3: break
        images = batch["image"].to(device)
        labels = batch["label"]
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, zero_division=0))

## 6. Discussion & Practical Considerations

### 6.1. Receptive Fields & Scale
AlexNet's massive $11 \times 11$ first conv layer has a large receptive field. It is designed to capture coarse features (like textures and broad color gradients) from large $224 \times 224$ images. Stride 4 sub-samples the image significantly, reducing memory and computation requirements down the line. Today, modern networks prefer stacking multiple $3 \times 3$ convolutions instead of large $11 \times 11$ filters because it offers more non-linear representation capacity with far fewer parameters.

### 6.2. Dropout Efficacy
Without Dropout ($p=0.5$), the fully connected layers ($4096 \to 4096 \to 10$) would overfit within a few epochs. Dropout forces the network to learn redundant representations by randomly disabling activations during training, creating an ensemble-like averaging effect.